### **Read JSONL**

In [13]:
import json
import pandas as pd

data = []

with open(r"C:\Users\hende\datasets\FINAL_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except:
            continue

df = pd.DataFrame(data)

print(df.head())
print(df.columns)

                                               lines  \
0  [def _download_artifact(artifact_path):, """, ...   
1  [def get_window(, window: Union[str, Tuple[str...   
2  [from flask import (render_template, url_for, ...   
3  [def decode(string):, """, Decode a reference ...   
4  [def apply_xsl(mml, xsl):, """Apply a xsl to a...   

                                           raw_lines  \
0  [def _download_artifact(artifact_path):,     "...   
1  [def get_window(,     window: Union[str, Tuple...   
2  [from flask import (render_template, url_for, ...   
3  [def decode(string):,     """,     Decode a re...   
4  [def apply_xsl(mml, xsl):,     """Apply a xsl ...   

                                               label  \
0  [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
1  [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...   
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
3                              [0, 0, 0, 0, 0, 0, 0]   
4  [0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 

In [20]:
import ast

def extract_identifiers(code):
    identifiers = {
        "functions": [],
        "variables": [],
        "classes": []
    }

    try:
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                identifiers["functions"].append(node.name)

            elif isinstance(node, ast.ClassDef):
                identifiers["classes"].append(node.name)

            elif isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        identifiers["variables"].append(target.id)

    except:
        return {
            "functions": [],
            "variables": [],
            "classes": []
        }

    return identifiers

### **Extraction**

In [21]:
print(df.columns.tolist())

['lines', 'raw_lines', 'label', 'type', 'cwe_id', 'dataset_source', 'code']


In [22]:
df["code"] = df["raw_lines"].apply(lambda x: "\n".join(x) if isinstance(x, list) else "")

In [23]:
df["identifiers"] = df["code"].apply(extract_identifiers)

<unknown>:182: SyntaxWarning: invalid escape sequence '\('
<unknown>:203: SyntaxWarning: invalid escape sequence '\)'
<unknown>:69: SyntaxWarning: invalid escape sequence '\,'
<unknown>:139: SyntaxWarning: invalid escape sequence '\,'
<unknown>:199: SyntaxWarning: invalid escape sequence '\('
<unknown>:209: SyntaxWarning: invalid escape sequence '\('
<unknown>:168: SyntaxWarning: invalid escape sequence '\s'
<unknown>:170: SyntaxWarning: invalid escape sequence '\s'
<unknown>:38: SyntaxWarning: invalid escape sequence '\('
<unknown>:40: SyntaxWarning: invalid escape sequence '\/'
<unknown>:41: SyntaxWarning: invalid escape sequence '\/'
<unknown>:42: SyntaxWarning: invalid escape sequence '\/'
<unknown>:43: SyntaxWarning: invalid escape sequence '\/'
<unknown>:44: SyntaxWarning: invalid escape sequence '\/'
<unknown>:45: SyntaxWarning: invalid escape sequence '\/'
<unknown>:46: SyntaxWarning: invalid escape sequence '\/'
<unknown>:47: SyntaxWarning: invalid escape sequence '\/'
<unknow

### **Features**

In [25]:
df["functions"] = df["identifiers"].apply(lambda x: x["functions"])
df["variables"] = df["identifiers"].apply(lambda x: x["variables"])
df["classes"] = df["identifiers"].apply(lambda x: x["classes"])

In [26]:
df["num_functions"] = df["functions"].apply(len)
df["num_variables"] = df["variables"].apply(len)
df["num_classes"] = df["classes"].apply(len)

In [27]:
df["total_identifiers"] = (
    df["num_functions"] +
    df["num_variables"] +
    df["num_classes"]
)

In [28]:
print(df[["functions", "variables", "classes"]].head())

                                      functions  \
0  [_download_artifact, stream_and_remove_file]   
1                                  [get_window]   
2                                            []   
3                                      [decode]   
4                                   [apply_xsl]   

                                           variables classes  
0  [tmp_dir, artifact_repo, dst, file_handle, fil...      []  
1  [sym, args, params, kwargs, winstr, winfunc, a...      []  
2                                                 []      []  
3                                           [string]      []  
4                     [s, transform, doc, result, s]      []  


In [29]:
df["code_length"] = df["code"].apply(len)

In [30]:
df.to_csv("final_dataset_ready.csv", index=False)

## **streamlit**

In [32]:
!pip install streamlit